# Regressão Linear Múltipla — Exemplo 03

Prever o **valor total da nota fiscal** usando:
- quantidade total de itens
- quantidade de produtos distintos
- forma de pagamento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

## 1. Conexão com o Banco de Dados PostgreSQL

In [ ]:
usuario = "datadt_data_analytics"
senha = "DataAnalytics$100"
host = "postgresql-datadt.alwaysdata.net"
porta = "5432"
banco = "datadt_digital_corporativo"

engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
)

## 2. Consulta SQL

Cada linha representa uma nota fiscal.

| Variável | Coluna |
|----------|--------|
| Y | valor_total_nota |
| X1 | quantidade_total_itens |
| X2 | quantidade_produtos_distintos |
| X3 | forma_pagamento |

In [ ]:
sql = """
SELECT 
    nf.id AS id_nota_fiscal,
    fp.descricao AS forma_pagamento,
    SUM(inf.quantidade) AS quantidade_total_itens,
    COUNT(DISTINCT inf.id_produto) AS quantidade_produtos_distintos,
    SUM(inf.quantidade * inf.valor_unitario) AS valor_total_nota
FROM vendas.nota_fiscal nf
JOIN vendas.item_nota_fiscal inf 
    ON inf.id_nota_fiscal = nf.id
JOIN vendas.forma_pagamento fp 
    ON fp.id = nf.id_forma_pagto
GROUP BY 
    nf.id,
    fp.descricao
ORDER BY nf.id;
"""

## 3. Carregando os Dados

In [ ]:
df = pd.read_sql(sql, engine)

print("Primeiras linhas da base:")
print(df.head())

print("\nInformações da base:")
print(df.info())

print("\nResumo estatístico:")
print(df[[
    "quantidade_total_itens",
    "quantidade_produtos_distintos",
    "valor_total_nota"
]].describe())

print("\nFormas de pagamento encontradas:")
print(df["forma_pagamento"].value_counts())

## 4. Tratamento Básico dos Dados

In [ ]:
df = df.dropna(subset=[
    "forma_pagamento",
    "quantidade_total_itens",
    "quantidade_produtos_distintos",
    "valor_total_nota"
])

df = df[
    (df["quantidade_total_itens"] > 0) &
    (df["quantidade_produtos_distintos"] > 0) &
    (df["valor_total_nota"] > 0)
]

print("Quantidade de registros após tratamento:")
print(len(df))

## 5. Definindo X e Y

In [ ]:
X = df[[
    "quantidade_total_itens",
    "quantidade_produtos_distintos",
    "forma_pagamento"
]]

y = df["valor_total_nota"]

## 6. Identificando Variáveis Numéricas e Categóricas

In [ ]:
variaveis_numericas = [
    "quantidade_total_itens",
    "quantidade_produtos_distintos"
]

variaveis_categoricas = [
    "forma_pagamento"
]

## 7. Pré-processamento

`OneHotEncoder` transforma a forma de pagamento em colunas numéricas binárias.

Exemplo:
- `forma_pagamento_Cartão` = 1 ou 0
- `forma_pagamento_Dinheiro` = 1 ou 0
- `forma_pagamento_Pix` = 1 ou 0

In [ ]:
pre_processador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variaveis_categoricas
        ),
        (
            "numericas",
            "passthrough",
            variaveis_numericas
        )
    ]
)

print("Pré-processador criado:")
print(pre_processador)

## 8. Criando o Pipeline do Modelo

In [ ]:
modelo = Pipeline(
    steps=[
        ("pre_processador", pre_processador),
        ("regressao", LinearRegression())
    ]
)

## 9. Divisão em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 10. Treinando o Modelo

In [ ]:
modelo.fit(X_train, y_train)

## 11. Fazendo Previsões

In [ ]:
y_pred = modelo.predict(X_test)

resultado = pd.DataFrame({
    "quantidade_total_itens": X_test["quantidade_total_itens"],
    "quantidade_produtos_distintos": X_test["quantidade_produtos_distintos"],
    "forma_pagamento": X_test["forma_pagamento"],
    "valor_real": y_test,
    "valor_previsto": y_pred
})

print("Comparação entre valor real e valor previsto:")
print(resultado.head(10))

## 12. Avaliação do Modelo

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Métricas de avaliação:")
print(f"MAE  - Erro médio absoluto: {mae:.2f}")
print(f"MSE  - Erro quadrático médio: {mse:.2f}")
print(f"R²   - Coeficiente de determinação: {r2:.4f}")

## 13. Coeficientes do Modelo

Peso de cada variável no modelo.

In [ ]:
regressao = modelo.named_steps["regressao"]
pre_processador_treinado = modelo.named_steps["pre_processador"]

nomes_variaveis_categoricas = (
    pre_processador_treinado
    .named_transformers_["categoricas"]
    .get_feature_names_out(variaveis_categoricas)
)

nomes_variaveis = list(nomes_variaveis_categoricas) + variaveis_numericas

coeficientes = pd.DataFrame({
    "variavel": nomes_variaveis,
    "coeficiente": regressao.coef_
})

print("Intercepto do modelo:")
print(f"{regressao.intercept_:.2f}")

print("\nCoeficientes do modelo:")
print(coeficientes.sort_values(by="coeficiente", ascending=False))

## 14. Gráfico: Valor Real x Valor Previsto

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(y_test, y_pred, alpha=0.5, label="Notas fiscais")

plt.xlabel("Valor real da nota fiscal")
plt.ylabel("Valor previsto da nota fiscal")
plt.title("Regressão Linear Múltipla - Valor Real x Valor Previsto")
plt.legend()
plt.grid(True)

plt.savefig("grafico_regressao_exemplo03_real_vs_previsto.png", dpi=300, bbox_inches="tight")
plt.show()

print("Gráfico salvo em: grafico_regressao_exemplo03_real_vs_previsto.png")

## 15. Gráfico: Análise dos Erros

In [ ]:
erros = y_test - y_pred

plt.figure(figsize=(10, 6))

plt.scatter(y_pred, erros, alpha=0.5, label="Erros")
plt.axhline(y=0, linestyle="--", label="Erro zero")

plt.xlabel("Valor previsto")
plt.ylabel("Erro")
plt.title("Análise dos Erros da Regressão")
plt.legend()
plt.grid(True)

plt.savefig("grafico_regressao_exemplo03_erros.png", dpi=300, bbox_inches="tight")
plt.show()

print("Gráfico salvo em: grafico_regressao_exemplo03_erros.png")

## 16. Simulação de Previsão

Ajuste a forma de pagamento conforme os valores existentes na base.

In [ ]:
print("Formas de pagamento disponíveis na base:")
print(df["forma_pagamento"].unique())

nova_nota = pd.DataFrame({
    "quantidade_total_itens": [10],
    "quantidade_produtos_distintos": [4],
    "forma_pagamento": ["Dinheiro"]
})

valor_estimado = modelo.predict(nova_nota)

print("\nSimulação:")
print("Dados da nova nota fiscal:")
print(nova_nota)
print(f"\nValor estimado da nota fiscal: R$ {valor_estimado[0]:.2f}")